# Homework4 (DQN for CartPole-v1)

在此次中，你将亲手实现一个经典的深度 Q 网络 (Deep Q-Network, DQN) 算法，并用它来解决 OpenAI Gym 中的 `CartPole-v1` 环境。

## 🎯 作业目标

你的任务是补全代码中所有标记为 `【请在这里补全代码 X】` 的部分。

你需要补全的核心模块包括：

1.  **`DQNNetwork` (代码 1, 2):**
    * `__init__`: 定义神经网络的结构。
    * `forward`: 实现神经网络的前向传播。

2.  **`ReplayBuffer` (代码 3, 4):**
    * `push`: 将经验元组存入缓冲区。
    * `sample`: 从缓冲区中随机采样一批数据。

3.  **`DQNAgent` (代码 5, 6):**
    * `select_action`: 实现 ε-贪婪 (Epsilon-Greedy) 策略。
    * `train`: 实现 DQN 的核心训练逻辑，包括计算目标 Q 值和损失函数。

4.  **`train_dqn` (代码 7):**
    * 补全主训练循环，实现智能体与环境的交互。

## 📝 提交要求

请在补全所有代码后，从头到尾运行整个 Notebook，确保所有单元格都能正常运行，并能看到最终的训练日志和生成的 `cartpole_agent.gif` 动图。

In [1]:
# DQN (Deep Q-Network) 实现 CartPole-v1 游戏
# 
# 这个项目实现了经典的 DQN 算法来解决 OpenAI Gym 的 CartPole-v1 环境。
# 
# 主要特性：
# - 经验回放 (Experience Replay)
# - 目标网络 (Target Network)
# - ε-贪婪策略 (Epsilon-Greedy)
import warnings                                                                           # 导入 warnings 警告管理模块

warnings.filterwarnings("ignore", category=UserWarning, message="pkg_resources is deprecated")  # 忽略 pkg_resources 弃用警告，让输出更干净
import gymnasium as gym                                                                    # 导入 Gymnasium 强化学习环境库（提供 CartPole-v1）
import numpy as np                                                                         # 导入 NumPy 数值计算库，别名 np
import torch                                                                               # 导入 PyTorch 深度学习框架
import torch.nn as nn                                                                      # 导入 PyTorch 神经网络模块，别名 nn
import torch.optim as optim                                                                # 导入 PyTorch 优化器模块，别名 optim
from collections import deque                                                              # 导入双端队列 deque，用于实现经验回放缓冲区
import random                                                                              # 导入 random 随机数模块，用于采样和探索
import matplotlib.pyplot as plt                                                           # 导入 matplotlib 绘图库，用于可视化训练曲线
from datetime import datetime                                                              # 导入 datetime 时间模块，用于 TensorBoard 日志命名
import os,time                                                                             # 导入 os 操作系统接口和 time 时间模块
from PIL import Image                                                                      # 导入 Pillow 图像处理库，用于处理图像帧
import imageio                                                                             # 导入 imageio 库，用于把图像帧合成为 GIF 动图

# 配置参数 (保留)
config = {                                                                                 # 用一个字典集中管理所有超参数
    'gamma': 0.99,                                                                         # 折扣因子：未来奖励的重要程度（越接近 1 越看重未来）
    'epsilon_start': 1.0,                                                                  # 初始探索率：一开始 100% 随机探索
    'epsilon_min': 0.001,                                                                  # 最小探索率：最低保留 0.1% 的探索
    'epsilon_decay': 0.995,                                                                # 探索率衰减系数：每训练一步乘以 0.995
    'learning_rate': 0.005,                                                                # 学习率：优化器更新步长
    'batch_size': 64,                                                                      # 批次大小：每次从缓冲区采样的样本数量
    'buffer_size': 20000,                                                                  # 经验回放缓冲区容量：最多存储 2 万条经验
    'target_update_freq': 10                                                               # 目标网络更新频率：每训练 10 步同步一次
}                                                                                          # config 字典定义结束

## 注意

如果你没有 Pillow 和 imageio 库，可以在命令行cmd 中用如下指令安装

`pip install imageio Pillow -i https://pypi.tuna.tsinghua.edu.cn`

如果 你没有安装 CartPole-v1, 可以使用如下指令安装

`pip install "gymnasium[classic-control]" -i https://pypi.tuna.tsinghua.edu.cn`

In [2]:
class DQNNetwork(nn.Module):                                                             # 定义 DQN 神经网络类，继承 PyTorch 的 nn.Module
    # DQN 神经网络
    # 输入: 状态向量 (4维: 位置, 速度, 角度, 角速度)
    # 输出: 每个动作的Q值 (2维: 左移, 右移)
    def __init__(self, state_size, action_size, hidden_size=256):                        # 构造函数：接收状态维度、动作维度、隐藏层大小（默认256）
        super(DQNNetwork, self).__init__()                                               # 调用父类 nn.Module 的构造函数完成初始化
        # ==================== 【请在这里补全代码 1】 ====================
        # 目标: 定义一个三层的全连接神经网络 (MLP)
        #     - 隐藏层激活函数使用 ReLU
        #     - 结构: state_size -> hidden_size -> hidden_size -> action_size

        # 1. 定义第一个全连接层 (fc1)
        # 提示: 使用 nn.Linear()
        self.fc1=nn.Linear(state_size,hidden_size)                                                                                 # 第1层全连接：输入 state_size 维 → 输出 hidden_size 维

        # 2. 定义第二个全连接层 (fc2)
        self.fc2=nn.Linear(hidden_size,hidden_size)                                                                                 # 第2层全连接：输入 hidden_size 维 → 输出 hidden_size 维

        # 3. 定义输出层 (fc3)
        self.fc3=nn.Linear(hidden_size,action_size)                                                                                 # 第3层全连接（输出层）：输入 hidden_size 维 → 输出 action_size 维
        # =========================================================

    def forward(self, x):                                                                # 前向传播函数：定义数据在网络中的流向
        # 前向传播
        # ==================== 【请在这里补全代码 2】 ====================
        # 目标: 完成前向传播
        # 1. 通过 fc1 并应用 ReLU 激活函数
        # 提示: 使用 torch.relu()
        x=self.fc1(x)
        # 通过第1层全连接，并用 ReLU 激活函数引入非线性
        x=torch.relu(x)
        # 2. 通过 fc2 并应用 ReLU 激活函数
        x=self.fc2(x)                                                                                  # 通过第2层全连接，并再次用 ReLU 激活
        x=torch.relu(x)
        # 3. 通过 fc3 (输出层，Q值不需要激活函数)
        x=self.fc3(x)                                                                                 # 通过输出层，Q 值是连续数值，不需要激活函数
        # =========================================================
        return x                                                                         # 返回每个动作对应的 Q 值


class ReplayBuffer:                                                                      # 定义经验回放缓冲区类
    # 经验回放缓冲区
    # 存储 (state, action, reward, next_state, done) 元组
    def __init__(self, capacity):                                                        # 构造函数：接收缓冲区容量
        self.buffer = deque(maxlen=capacity)                                             # 用固定长度双端队列存储经验，超出容量自动丢弃最旧数据

    def push(self, state, action, reward, next_state, done):                             # 定义"存入一条经验"的方法
        # 添加一条经验到缓冲区
        # ==================== 【请在这里补全代码 3】 ====================
        # 目标: 将 (state, action, reward, next_state, done) 元组添加到缓冲区
        # 提示: self.buffer 是一个 deque 对象，使用 .append() 方法
        # =========================================================
        self.buffer.append((state, action, reward, next_state, done))                   # 把 (s,a,r,s',done) 打包成元组追加到队列尾部

    def sample(self, batch_size):                                                        # 定义"随机采样一批经验"的方法
        # 随机采样一批经验
        # ==================== 【请在这里补全代码 4】 ====================
        # 目标: 从 self.buffer 中随机采样 batch_size 个经验
        # 提示: 使用 random.sample(self.buffer, batch_size)
        batch=random.sample(self.buffer, batch_size)                                                                                 # 从缓冲区随机抽取 batch_size 条不重复的经验
        states, actions, rewards, next_states, dones = zip(*batch)
        # 目标: 将采样到的 'batch' (一个元组列表) 解压缩 (unzip) 为
        #       独立的 states, actions, rewards, next_states, dones 列表
        # 提示: 使用 zip(*batch)
                                                                                         # 用 zip(*batch) 把"经验列表"按列解压成 5 个独立元组
        # =========================================================

        # 返回值已保留
        return (np.array(states), np.array(actions), np.array(rewards),                  # 返回：把 5 个元组转成 NumPy 数组（状态、动作、
                np.array(next_states), np.array(dones))                                  #       奖励、下一状态、结束标志数组）

    def __len__(self):                                                                   # 定义获取缓冲区当前长度的魔法方法
        return len(self.buffer)                                                          # 返回当前已存储的经验条数


class DQNAgent:                                                                          # 定义 DQN 智能体类
    # DQN 智能体
    def __init__(self, state_size, action_size, config):                                 # 构造函数：接收状态维度、动作维度、超参数字典
        # (保留) 初始化 DQN 智能体
        self.state_size = state_size                                                     # 保存状态维度（CartPole 为 4）
        self.action_size = action_size                                                   # 保存动作维度（CartPole 为 2）
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")       # 自动选择计算设备：有 GPU 用 cuda，否则用 cpu

        # 超参数 (保留)
        self.gamma = config['gamma']                                                     # 从配置读取折扣因子
        self.epsilon = config['epsilon_start']                                           # 初始化探索率为配置的初始值
        self.epsilon_min = config['epsilon_min']                                         # 读取最小探索率
        self.epsilon_decay = config['epsilon_decay']                                     # 读取探索率衰减系数
        self.learning_rate = config['learning_rate']                                     # 读取学习率
        self.batch_size = config['batch_size']                                           # 读取批次大小
        self.target_update_freq = config['target_update_freq']                           # 读取目标网络更新频率

        # 创建策略网络和目标网络 (保留)
        self.policy_net = DQNNetwork(state_size, action_size).to(self.device)            # 创建策略网络（用于选动作和训练），搬到指定设备
        self.target_net = DQNNetwork(state_size, action_size).to(self.device)            # 创建目标网络（用于计算稳定的目标 Q 值）
        self.target_net.load_state_dict(self.policy_net.state_dict())                    # 初始时把策略网络的参数完整复制给目标网络
        self.target_net.eval()                                                           # 将目标网络设为评估模式（不更新参数、无 dropout）

        # 优化器和损失函数 (保留)
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=self.learning_rate) # 用 Adam 优化器优化策略网络参数
        self.criterion = nn.MSELoss()                                                    # 用均方误差 (MSE) 作为损失函数

        # 经验回放缓冲区 (保留)
        self.memory = ReplayBuffer(config['buffer_size'])                                # 创建经验回放缓冲区

        # 训练步数计数器 (保留)
        self.train_step = 0                                                              # 初始化训练步数计数器为 0

    def select_action(self, state, training=True):                                       # 定义"选择动作"的方法，training 控制是否探索
        # 使用 ε-贪婪策略选择动作
        # ==================== 【请在这里补全代码 5】 ====================
        # 目标: 实现 ε-贪婪策略

        # 1. 生成一个 (0, 1) 之间的随机数
        # 提示: 使用 random.random()
        k=random.random()                                                                                 # 生成一个 (0,1) 之间的随机数，用于决定探索还是利用

        # 2. 判断是否进行探索 (exploration)
        #    (条件: training 模式开启 且 随机数 < self.epsilon)
        if training and k < self.epsilon:                                                                                 # 若处于训练模式且随机数小于探索率 ε → 进入探索
            # 探索: 随机选择一个动作
            # 提示: self.action_size 是动作空间的大小, 使用 random.randrange()
            action=random.randrange(self.action_size)                                                              # 探索：随机选择一个动作（返回 0 或 1）

        # 3. 否则, 进行利用 (exploitation)
        else:                                                                            # 否则进入利用阶段
            # 利用: 选择Q值最大的动作
            # (上下文管理器 'with torch.no_grad():' 已保留)
            # 关闭梯度计算，加快推理速度、节省显存
                # (张量转换已保留)
            with torch.no_grad():

                # 把状态转成张量，并在第 0 维增加 batch 维度，搬到设备
                state=torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(self.device)
                # 3.1 使用 policy_net 预测 Q 值
                Q=self.policy_net(state)
                # 用策略网络预测该状态下所有动作的 Q 值

                # 3.2 选择 Q 值最大的动作
                action=Q.argmax(dim=1).item()
                # 提示: 使用 .argmax().item()
                                                                                         # 返回 Q 值最大的那个动作（argmax 取下标，item 转成 Python 数）
        # =========================================================
        return action                                                                        # 返回 Q 值最大的那个动作

    def train(self):                                                                     # 定义"训练网络"的方法（一次梯度更新）
        # 从经验回放缓冲区采样并训练网络
        # (缓冲区大小检查已保留)
        if len(self.memory) < self.batch_size:                                           # 若缓冲区数据不足一个 batch，无法采样训练
            return None                                                                  # 直接返回 None，跳过本次训练

        # (从缓冲区采样已保留)
        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)  # 从缓冲区随机采样一个 batch 的经验

        # (转换为 PyTorch 张量已保留)
        states = torch.FloatTensor(states).to(self.device)                               # 把状态数组转成浮点张量并搬到设备
        actions = torch.LongTensor(actions).to(self.device)                              # 把动作数组转成长整型张量（动作是离散下标）
        rewards = torch.FloatTensor(rewards).to(self.device)                             # 把奖励数组转成浮点张量
        next_states = torch.FloatTensor(next_states).to(self.device)                     # 把下一状态数组转成浮点张量
        dones = torch.FloatTensor(dones).to(self.device)                                 # 把结束标志数组转成浮点张量（用于乘 (1-done)）

        # ==================== 【请在这里补全代码 6】 ====================

        # 1. 计算当前 Q 值 (current_q_values)
        #    - 使用 self.policy_net 预测所有状态的 Q(s, a)
        #    - 从中 "收集" (gather) 出实际采取的 'actions' 对应的 Q 值
        # 提示:
        all_q_values = self.policy_net(states)
        current_q_values = all_q_values.gather(1, actions.unsqueeze(1)).squeeze(1)
        # 用策略网络计算 batch 中每个状态在所有动作上的 Q 值
        # 用 gather 取出"实际执行动作"对应的 Q 值（当前 Q 值）
        # 2. 计算目标 Q 值 (target_q_values)
        # (上下文管理器 'with torch.no_grad():' 已保留)
        with torch.no_grad():                                                            # 计算目标 Q 值时关闭梯度（目标网络不更新）
            # 2.1 使用 self.target_net 预测 next_states 的 Q 值
            # 并找出每个 next_state 的最大 Q 值 (Q_next)
            # 提示: 使用 .max(1)[0]
            next_q_values=self.target_net(next_states)                                                                             # 用目标网络计算下一状态的所有动作 Q 值
            Q_next=next_q_values.max(1)[0]                                  # 取每个下一状态的最大 Q 值（即 max_a Q(s',a)）
            # 2.2 根据贝尔曼方程计算目标 Q 值
            #     target = reward + gamma * Q_next * (1 - done)
            target=rewards+self.gamma* Q_next*(1-dones)                         # 贝尔曼方程算目标值：r + γ·maxQ(s')·(1-done)

        # 3. 计算损失 (Loss)
        #    - 计算 'current_q_values' 和 'target_q_values' 之间的均方误差 (MSE)
        # 提示: 使用 self.criterion()
        loss=self.criterion(current_q_values, target)
        # 计算当前 Q 值与目标 Q 值之间的均方误差损失

        # =========================================================

        # (反向传播和优化已保留)
        self.optimizer.zero_grad()                                                       # 清空上一步的梯度，防止梯度累加
        loss.backward()                                                                  # 反向传播，计算各参数的梯度
        self.optimizer.step()                                                            # 用优化器根据梯度更新策略网络参数

        # (更新训练步数已保留)
        self.train_step += 1                                                             # 训练步数 +1

        # (定期更新目标网络已保留)
        if self.train_step % self.target_update_freq == 0:                               # 每训练 target_update_freq 步（默认 10 步）
            self.target_net.load_state_dict(self.policy_net.state_dict())                # 把策略网络的参数复制给目标网络（同步目标网络）

        # (衰减探索率已保留)
        if self.epsilon > self.epsilon_min:                                              # 若探索率还没降到最小值
            self.epsilon *= self.epsilon_decay                                           # 按衰减系数递减探索率（逐步减少随机探索）

        return loss.item()                                                               # 返回本次损失的数值（用于记录日志）

    def save(self, filepath):                                                            # 定义"保存模型"的方法
        # (保留) 保存模型
        torch.save({                                                                     # 用 torch.save 把训练状态打包保存
            'policy_net_state_dict': self.policy_net.state_dict(),                       # 保存策略网络权重
            'target_net_state_dict': self.target_net.state_dict(),                       # 保存目标网络权重
            'optimizer_state_dict': self.optimizer.state_dict(),                         # 保存优化器状态
            'epsilon': self.epsilon,                                                     # 保存当前探索率
            'train_step': self.train_step                                                # 保存当前训练步数
        }, filepath)                                                                     # 写入指定文件路径
        print(f"模型已保存到 {filepath}")                                                 # 打印保存成功提示

    def load(self, filepath):                                                            # 定义"加载模型"的方法
        # (保留) 加载模型
        checkpoint = torch.load(filepath, map_location=self.device)                      # 从文件加载检查点，并映射到当前设备
        self.policy_net.load_state_dict(checkpoint['policy_net_state_dict'])             # 恢复策略网络权重
        self.target_net.load_state_dict(checkpoint['target_net_state_dict'])             # 恢复目标网络权重
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])               # 恢复优化器状态
        self.epsilon = checkpoint['epsilon']                                             # 恢复探索率
        self.train_step = checkpoint['train_step']                                       # 恢复训练步数
        print(f"模型已从 {filepath} 加载")                                                # 打印加载成功提示


In [3]:
def train_dqn(episodes=800, render=False, config=config):                               # 定义训练函数：默认训练 800 回合
    # (保留大部分) 训练 DQN 智能体

    # (创建环境和智能体已保留)
    env = gym.make('CartPole-v1', render_mode='human' if render else None)               # 创建 CartPole-v1 环境（render=True 时才弹出可视化窗口）
    state_size = env.observation_space.shape[0]                                          # 从环境读取状态维度（= 4）
    action_size = env.action_space.n                                                     # 从环境读取动作数量（= 2）
    agent = DQNAgent(state_size, action_size, config=config)                             # 根据状态/动作维度创建 DQN 智能体

    # (TensorBoard 设置已保留)
    writer = None                                                                        # 初始化 TensorBoard 写入器为 None
    tensorboard_vis = False                                                              # 默认关闭 TensorBoard 可视化
    if tensorboard_vis:                                                                  # 若开启 TensorBoard 可视化（此处为 False，不执行）
        try:                                                                             # 尝试导入 TensorBoard
            from torch.utils.tensorboard import SummaryWriter                            # 导入 SummaryWriter 写入器
            log_dir = f"runs/DQN_CartPole_{datetime.now().strftime('%Y%m%d_%H%M%S')}"    # 用时间戳生成日志目录名
            writer = SummaryWriter(log_dir)                                              # 创建 SummaryWriter 写入器
            print(f"TensorBoard 日志保存到: {log_dir}")                                   # 打印日志保存目录
            print(f"运行命令查看: tensorboard --logdir=runs")                             # 打印查看命令
        except ImportError:                                                              # 若未安装 TensorBoard
            print("警告: 未安装 TensorBoard，跳过可视化")                                  # 打印警告并跳过
            tensorboard_vis = False                                                      # 关闭 TensorBoard 开关

    # (训练统计变量已保留)
    episode_rewards = []                                                                 # 列表：记录每个回合的总奖励
    episode_losses = []                                                                  # 列表：记录每个回合的平均损失
    moving_avg_rewards = []                                                              # 列表：记录滑动平均奖励（近 100 回合）

    print("开始训练...")                                                                  # 打印训练开始提示
    print(f"设备: {agent.device}")                                                       # 打印当前计算设备

    # (主循环已保留)
    for episode in range(episodes):                                                      # 外层循环：遍历每一个训练回合
        state, _ = env.reset()                                                           # 重置环境，得到初始状态 state
        episode_reward = 0                                                               # 初始化本回合累计奖励为 0
        episode_loss = []                                                                # 初始化本回合损失列表为空
        done = False                                                                     # 初始化"是否结束"标志为 False

        while not done:                                                                  # 内层循环：一个回合内不断与环境交互，直到结束

            # ==================== 【请在这里补全代码 7】 ====================
            # 1. 选择动作
            # 提示: 使用 agent.select_action(state)
            action=agent.select_action(state)                                                             # 第1步：用 ε-贪婪策略选择动作

            # 2. 在环境中执行动作
            # 提示: 使用 env.step(action)
            next_state, reward, terminated, truncated, _ = env.step(action)                # 第2步：在环境中执行动作，得到下一状态/奖励/结束标志
            done = terminated or truncated                                               # 任一终止或截断标志为真，都视为回合结束

            # 3. 存储经验到回放缓冲区
            # 提示: 使用 agent.memory.push(...)
            # (注意: done 需要是 float 类型)
            agent.memory.push(state, action, reward, next_state, float(done))            # 第3步：把这条经验存入回放缓冲区

            # 4. 训练智能体
            # 提示: 调用 agent.train()
            loss = agent.train()                                                          # 第4步：从缓冲区采样并训练一次网络
            if loss is not None:                                                          # 若成功训练（返回了损失值）
                episode_loss.append(loss)                                                # 把本次损失记录到本回合损失列表

            # 5. 更新状态和奖励
            state = next_state                                                           # 第5步：把下一状态更新为当前状态
            episode_reward += reward                                                     # 累加本回合奖励
            # =========================================================

        # (记录统计信息已保留)
        episode_rewards.append(episode_reward)                                           # 记录本回合总奖励
        avg_loss = np.mean(episode_loss) if episode_loss else 0                          # 计算本回合平均损失（无损失则记为 0）
        episode_losses.append(avg_loss)                                                  # 记录本回合平均损失

        if len(episode_rewards) >= 100:                                                  # 若已积累至少 100 个回合
            moving_avg = np.mean(episode_rewards[-100:])                                 # 取最近 100 回合的平均奖励
        else:                                                                            # 否则
            moving_avg = np.mean(episode_rewards)                                        # 取目前所有回合的平均奖励
        moving_avg_rewards.append(moving_avg)                                            # 记录滑动平均奖励

        # (记录到 TensorBoard 已保留)
        if tensorboard_vis and writer:                                                   # 若开启了 TensorBoard 可视化
            writer.add_scalar('Training/Episode_Reward', episode_reward, episode)        # 记录本回合奖励到 TensorBoard
            writer.add_scalar('Training/Moving_Avg_Reward', moving_avg, episode)         # 记录滑动平均奖励
            writer.add_scalar('Training/Loss', avg_loss, episode)                        # 记录平均损失
            writer.add_scalar('Training/Epsilon', agent.epsilon, episode)                # 记录探索率

        # (打印进度已保留)
        if (episode + 1) % 10 == 0:                                                      # 每 10 个回合打印一次进度
            print(f"回合: {episode + 1}/{episodes}, "                                     # 打印回合序号
                  f"奖励: {episode_reward:.2f}, "                                         # 打印本回合奖励
                  f"平均奖励(100回合): {moving_avg:.2f}, "                                 # 打印滑动平均奖励
                  f"损失: {avg_loss:.4f}, "                                               # 打印平均损失
                  f"探索率: {agent.epsilon:.4f}")                                         # 打印当前探索率

        # (提前结束条件已保留)
        if moving_avg >= 475:                                                            # 若滑动平均奖励达到 475（判定为"解决"）
            print(f"\n环境已解决！在第 {episode + 1} 回合达到平均奖励 {moving_avg:.2f}")    # 打印解决提示
            break                                                                        # 提前结束训练

    # (关闭 TensorBoard 和保存模型已保留)
    if writer:                                                                           # 若 TensorBoard 写入器存在
        writer.close()                                                                   # 关闭写入器

    os.makedirs('models', exist_ok=True)                                                 # 确保 models 目录存在（不存在则创建）
    model_path = 'models/dqn_cartpole.pth'                                               # 定义模型保存路径
    agent.save(model_path)                                                               # 保存训练好的模型

    env.close()                                                                          # 关闭环境释放资源
    return agent, episode_rewards                                                        # 返回智能体和每回合奖励列表


def test_agent(episodes=10, render=True, config=config, gif_path="cartpole_agent.gif"):  # 定义测试函数：默认测试 10 回合并生成 GIF
    # (保留) 测试训练好的智能体并生成 GIF 动画
    # 创建环境（使用 rgb_array 渲染模式以便捕获帧）
    env = gym.make('CartPole-v1', render_mode='rgb_array')                               # 创建环境，用 rgb_array 渲染以便捕获每一帧图像

    state_size = env.observation_space.shape[0]                                          # 读取状态维度
    action_size = env.action_space.n                                                     # 读取动作数量

    agent = DQNAgent(state_size, action_size, config=config)                             # 创建一个新的智能体（稍后加载权重）

    # 确保模型文件存在，如果不存在则跳过加载
    model_path = 'models/dqn_cartpole.pth'                                               # 定义模型文件路径
    if not os.path.exists(model_path):                                                   # 若模型文件不存在
        print(f"警告: 找不到模型文件 {model_path}。请先运行训练。")                         # 打印警告
        print("跳过测试。")                                                               # 打印跳过提示
        env.close()                                                                      # 关闭环境
        return []                                                                        # 返回空列表

    agent.load(model_path)                                                               # 加载训练好的模型权重

    print("\n开始测试并录制 GIF...")                                                      # 打印测试开始提示
    test_rewards = []                                                                    # 列表：记录每个测试回合的奖励
    frames = []                                                                          # 列表：存储所有帧图像，用于合成 GIF（存储每一帧图像）

    for episode in range(episodes):                                                      # 遍历每个测试回合
        state, _ = env.reset()                                                           # 重置环境得到初始状态
        episode_reward = 0                                                               # 初始化本回合奖励
        done = False                                                                     # 初始化结束标志

        while not done:                                                                  # 内层循环：与环境交互直到回合结束
            # 使用贪婪策略（不探索）
            action = agent.select_action(state, training=False)                          # 用贪婪策略选动作（training=False 表示不探索）
            next_state, reward, terminated, truncated, _ = env.step(action)              # 执行动作，得到下一状态/奖励/结束标志
            done = terminated or truncated                                               # 更新结束标志

            state = next_state                                                           # 更新状态
            episode_reward += reward                                                     # 累加奖励

            # 捕获当前帧并添加到列表
            frame = env.render()                                                         # 渲染当前画面得到一帧图像（RGB 数组）
            frames.append(Image.fromarray(frame))                                        # 把数组帧转成 PIL 图像并加入帧列表


        test_rewards.append(episode_reward)                                              # 记录本回合奖励
        print(f"测试回合 {episode + 1}: 奖励 = {episode_reward}")                         # 打印本回合奖励

    # 保存为 GIF
    imageio.mimsave(                                                                     # 用 imageio 把帧序列合成 GIF 动图
        gif_path,                                                                        # GIF 保存路径
        [np.array(frame) for frame in frames],                                           # 把每帧 PIL 图像转回 NumPy 数组
        duration=1000 / 30                                                               # 每帧显示时长(毫秒)，约等于 30 帧/秒（新版 imageio 用 duration 取代 fps）
    )                                                                                    # mimsave 调用结束（合成并保存 GIF）
    print(f"GIF 已保存至: {gif_path}")                                                   # 打印 GIF 保存路径

    avg_reward = np.mean(test_rewards)                                                   # 计算测试平均奖励
    print(f"\n测试平均奖励: {avg_reward:.2f}")                                            # 打印测试平均奖励

    env.close()                                                                          # 关闭环境
    return test_rewards                                                                  # 返回各回合奖励列表


def main():                                                                              # 定义主函数
    # (保留) 主函数：训练和测试 DQN 智能体
    print("=" * 60)                                                                      # 打印分隔线
    print("DQN CartPole-v1 项目")                                                         # 打印项目标题
    print("=" * 60)                                                                      # 打印分隔线

    # 设置随机种子以确保可重复性
    seed = 42                                                                            # 设置随机种子值
    torch.manual_seed(seed)                                                              # 固定 PyTorch 随机种子（保证可重复）
    np.random.seed(seed)                                                                 # 固定 NumPy 随机种子
    random.seed(seed)                                                                    # 固定 Python 随机种子

    # 训练智能体
    agent, rewards = train_dqn(                                                          # 调用训练函数训练智能体
        render=False,                                                                    # 训练时不打开可视化窗口
        config=config                                                                    # 传入超参数字典
    )                                                                                    # train_dqn 调用结束

    print("模型保存在: models/dqn_cartpole.pth")                                          # 打印模型保存位置
    print("=" * 60)                                                                      # 打印分隔线


In [4]:
# 运行主函数开始训练
# 补全以上所有代码后，运行此单元格
# 说明：train_dqn 默认 800 回合；本机 CPU 上为快速演示这里传入 400 回合。
# 若要完全按原题跑满 800 回合，把下面这行改成 main() 即可。
agent, rewards = train_dqn(episodes=400, render=False, config=config)                    # 训练 DQN 智能体并得到每回合奖励


开始训练...
设备: cpu
回合: 10/400, 奖励: 191.00, 平均奖励(100回合): 62.10, 损失: 6.1139, 探索率: 0.0610
回合: 20/400, 奖励: 291.00, 平均奖励(100回合): 158.25, 损失: 201.9363, 探索率: 0.0010
回合: 30/400, 奖励: 142.00, 平均奖励(100回合): 169.30, 损失: 30.5064, 探索率: 0.0010
回合: 40/400, 奖励: 500.00, 平均奖励(100回合): 232.62, 损失: 40.9357, 探索率: 0.0010
回合: 50/400, 奖励: 260.00, 平均奖励(100回合): 258.52, 损失: 54.9708, 探索率: 0.0010
回合: 60/400, 奖励: 500.00, 平均奖励(100回合): 283.35, 损失: 70.8762, 探索率: 0.0010
回合: 70/400, 奖励: 379.00, 平均奖励(100回合): 296.87, 损失: 210.9219, 探索率: 0.0010
回合: 80/400, 奖励: 500.00, 平均奖励(100回合): 312.39, 损失: 129.3693, 探索率: 0.0010
回合: 90/400, 奖励: 500.00, 平均奖励(100回合): 316.11, 损失: 185.0698, 探索率: 0.0010
回合: 100/400, 奖励: 500.00, 平均奖励(100回合): 326.58, 损失: 152.6373, 探索率: 0.0010
回合: 110/400, 奖励: 261.00, 平均奖励(100回合): 363.82, 损失: 62.1297, 探索率: 0.0010
回合: 120/400, 奖励: 260.00, 平均奖励(100回合): 381.78, 损失: 73.2801, 探索率: 0.0010
回合: 130/400, 奖励: 107.00, 平均奖励(100回合): 395.94, 损失: 66.4548, 探索率: 0.0010
回合: 140/400, 奖励: 500.00, 平均奖励(100回合): 379.82, 损失: 176.9433, 探索率: 0.

In [5]:
# 运行测试函数
# 确保你已经成功运行了上面的 main() 并生成了 'models/dqn_cartpole.pth' 文件
test_agent(episodes=5, render=True, config=config)         # 测试 5 回合并录制 cartpole_agent.gif 动图


模型已从 models/dqn_cartpole.pth 加载

开始测试并录制 GIF...
测试回合 1: 奖励 = 500.0
测试回合 2: 奖励 = 500.0
测试回合 3: 奖励 = 500.0
测试回合 4: 奖励 = 500.0
测试回合 5: 奖励 = 500.0
GIF 已保存至: cartpole_agent.gif

测试平均奖励: 500.00


[500.0, 500.0, 500.0, 500.0, 500.0]